### TOOL BINDING

In [61]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [62]:
#tool create

@tool
def multiply(a:int, b:int) ->int:
    '''Given two number this tool returns thier product'''
    return a * b

In [63]:
multiply.invoke({'a':2, 'b':2})

4

In [64]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given two number this tool returns thier product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [65]:
#tool binding

model = ChatOpenAI(model='gpt-4o-mini') #only few llm can bind tools not all
model_with_tools= model.bind_tools([multiply])


### Tool calling

In [66]:
model_with_tools.invoke("Hi how are you?")

AIMessage(content="I'm just a computer program, so I don't have feelings, but I'm here and ready to assist you! How can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 55, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bda4d3a2c', 'id': 'chatcmpl-CE9S0RQF93HrCcruasUkNkWQ3A7nd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--45cdbfc6-9dfe-4c8f-bdc0-62ae4dd5a035-0', usage_metadata={'input_tokens': 55, 'output_tokens': 29, 'total_tokens': 84, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [67]:
query = HumanMessage("can you multiply three with ten?")

In [68]:
messages = [query]

In [69]:
messages

[HumanMessage(content='can you multiply three with ten?', additional_kwargs={}, response_metadata={})]

In [70]:
response = model_with_tools.invoke(messages) #only suggest the tool and input argument, but llm do not call tool and execute, the actual execution is handeled by you

In [71]:
messages.append(response)

In [72]:
messages

[HumanMessage(content='can you multiply three with ten?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_TDf8mMQVvRfJWwfLxWwtgHL8', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 57, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bda4d3a2c', 'id': 'chatcmpl-CE9S49jzCjzhmM3AustqolibmP8Oz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--1feece83-9c80-4772-a4bf-0e91d6ae0102-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_TDf8mMQVvRfJWwfLxWwtgHL8', 'type': 'tool_call'}], usage_metada

In [73]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 10},
  'id': 'call_TDf8mMQVvRfJWwfLxWwtgHL8',
  'type': 'tool_call'}]

In [74]:
response.tool_calls[0]


{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'call_TDf8mMQVvRfJWwfLxWwtgHL8',
 'type': 'tool_call'}

### TOOL EXECUTION

In [75]:
response.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'call_TDf8mMQVvRfJWwfLxWwtgHL8',
 'type': 'tool_call'}

In [76]:
response.tool_calls[0]['args']


{'a': 3, 'b': 10}

In [77]:
multiply.invoke(response.tool_calls[0]['args'])

30

In [78]:
tool_message = multiply.invoke(response.tool_calls[0])  #Toolmessege


In [79]:
messages.append(tool_message)

In [80]:
messages

[HumanMessage(content='can you multiply three with ten?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_TDf8mMQVvRfJWwfLxWwtgHL8', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 57, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8bda4d3a2c', 'id': 'chatcmpl-CE9S49jzCjzhmM3AustqolibmP8Oz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--1feece83-9c80-4772-a4bf-0e91d6ae0102-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_TDf8mMQVvRfJWwfLxWwtgHL8', 'type': 'tool_call'}], usage_metada

### Now how LLm will send the final output

In [81]:
model_with_tools.invoke(messages).content

'The product of three and ten is 30.'